# 实验三：Stage 1 × Stage 2（S2-C）交叉过滤

**逻辑**：
1. 读取 Stage 1 结果 → 获取每篇文章预测的技术集合
2. 从 S3 读取 S2-C 预测 span
3. 过滤：只保留 Stage 1 也预测了该技术的 span
4. 与金标准对比，计算 Span F1

**组合**：
- Combo-1: Sup-FT × S2-C
- Combo-2: Prompt-A × S2-C
- Combo-3: Iter-Ens × S2-C

In [ ]:
# ============================================================
# CONFIGURATION — update these paths before running
# ============================================================
import os

# Root directory containing data/, results/, technique_list/, etc.
BASE = "your/path/here"  # e.g. "/home/user/propaganda-span-detection"

# SemEval-2023 / CheckThat!-2024 overlap evaluation set
# Expected subdirectories: en_overlap_articles/, po_overlap_articles/, ru_overlap_articles/
OVERLAP_DIR = "your/path/here"  # typically BASE + "/data_analy/overlap_analysis_results"

# CheckThat! 2024 Task 3 training data bundle
CHECKTHAT24_DIR = "your/path/here"  # root of the official CheckThat!-2024 data bundle

# S3 credentials (or set as environment variables)
os.environ.setdefault("AWS_ACCESS_KEY_ID",     "your-access-key-id")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")
S3_ENDPOINT = "https://your-s3-endpoint"
S3_BUCKET   = "your-s3-bucket"


In [ ]:
# ============================================================
# Cell 1：导入库
# ============================================================
import io
import os
import boto3
import pandas as pd
from botocore.client import Config
from collections import defaultdict

print('✅ Libraries imported')

In [ ]:
# ============================================================
# Cell 2：路径配置
# ============================================================

BASE = "your/path/here"
GOLD_BASE = "your/path/here"

# Stage 1 文件路径 + 是否有 article 前缀
STAGE1_FILES = {
    'Sup-FT': {
        'EN': (BASE + 'model_tests/semeval_task3_en_dev_predictions_official_format.txt', True),
        'PL': (BASE + 'model_tests/semeval_task3_po_dev_predictions_official_format.txt', True),
        'RU': (BASE + 'model_tests/semeval_task3_ru_dev_predictions_official_format.txt', True),
    },
    'Prompt-A': {
        'EN': (BASE + 'LLM_tests/results_en_results_semeval_task3_dev_en_context_all.tsv', False),
        'PL': (BASE + 'LLM_tests/semeval2023_task3_dev_results_po_po_context_all.tsv', False),
        'RU': (BASE + 'LLM_tests/semeval2023_task3_dev_results_ru_ru_context_all.tsv', False),
    },
    'Iter-Ens': {
        'EN': (BASE + 'LLM_tests/results_en_results_semeval_task3_dev_en_voting_aggressive_all.tsv', False),
        'PL': (BASE + 'LLM_tests/semeval2023_task3_dev_results_po_po_voting_aggressive_all.tsv', False),
        'RU': (BASE + 'LLM_tests/semeval2023_task3_dev_results_ru_ru_voting_aggressive_all.tsv', False),
    },
}

# 金标准路径
GOLD_FILES = {
    'EN': GOLD_BASE + 'en_overlap_articles/labels-subtask-3-spans.txt',
    'PL': GOLD_BASE + 'po_overlap_articles/labels-subtask-3-spans.txt',
    'RU': GOLD_BASE + 'ru_overlap_articles/labels-subtask-3-spans.txt',
}

# S2-C 预测文件（S3 key）
S2C_S3_KEYS = {
    'EN': 'multi_label_models/exp1_s2c_mbert_l10_multilingual/predictions_en.tsv',
    # 'PL': 'multi_label_models/exp1_s2c_mbert_l10_multilingual/predictions_pl.tsv',
    'PL': 'multi_label_models/exp1_s2c_mbert_l10_pl_fixed/predictions_pl.tsv',
    'RU': 'multi_label_models/exp1_s2c_mbert_l10_multilingual/predictions_ru.tsv',
}

# 路径验证
print('=== Stage 1 文件验证 ===')
for s1, langs in STAGE1_FILES.items():
    for lang, (path, _) in langs.items():
        status = '✅' if os.path.exists(path) else '❌'
        print(f'  {status} {s1} {lang}: {os.path.basename(path)}')

print('\n=== 金标准文件验证 ===')
for lang, path in GOLD_FILES.items():
    status = '✅' if os.path.exists(path) else '❌'
    print(f'  {status} {lang}: {os.path.basename(path)}')

In [ ]:
# ============================================================
# Cell 3：S3 配置 + 加载 S2-C 预测
# ============================================================
ENDPOINT              = "https://your-s3-endpoint"  # Set to your S3-compatible endpoint
S3_BUCKET             = "your-s3-bucket"
AWS_ACCESS_KEY_ID = os.environ.get("AWS_ACCESS_KEY_ID", "your-access-key-id")
AWS_SECRET_ACCESS_KEY = os.environ.get("AWS_SECRET_ACCESS_KEY", "your-secret-access-key")

s3 = boto3.client(
    's3', endpoint_url=ENDPOINT,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    config=Config(signature_version='s3v4'),
)

def load_s2c_from_s3(s3_key):
    """从 S3 读取 S2-C predictions TSV，标准化 article_id 为纯数字"""
    buf = io.BytesIO()
    s3.download_fileobj(S3_BUCKET, s3_key, buf)
    buf.seek(0)
    df = pd.read_csv(buf, sep='\t')
    df['article_id'] = df['article_id'].astype(str).str.replace('article', '', regex=False)
    return df

# 加载三语言 S2-C 预测
s2c_preds = {}
for lang, key in S2C_S3_KEYS.items():
    s2c_preds[lang] = load_s2c_from_s3(key)
    print(f'✅ S2-C {lang}: {len(s2c_preds[lang])} spans')

In [ ]:
# ============================================================
# Cell 4：工具函数定义
# ============================================================

def load_stage1(path, has_article_prefix):
    """
    读取 Stage 1 结果（段落级预测）
    返回 { article_id(纯数字): set(technique, ...) }
    格式：article_id  technique  start  end（无表头，\t分隔）
    """
    tech_by_art = defaultdict(set)
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 4:
                continue
            art_id = parts[0].replace('article', '')
            technique = parts[1]
            tech_by_art[art_id].add(technique)
    return tech_by_art


def filter_s2c_by_stage1(s2c_df, stage1_tech_by_art):
    """
    过滤 S2-C 预测：只保留 Stage 1 也预测了该技术的 span
    """
    def keep(row):
        art_techs = stage1_tech_by_art.get(str(row['article_id']), set())
        return row['technique'] in art_techs
    mask = s2c_df.apply(keep, axis=1)
    return s2c_df[mask].copy()


def load_gold(path):
    """
    读取金标准
    格式：article813452859  technique  start  end（无表头，4列）
    返回 [(article_id, technique, start, end), ...]
    """
    spans = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 4:
                continue
            art_id = parts[0].replace('article', '')
            spans.append((art_id, parts[1], int(parts[2]), int(parts[3])))
    return spans


def compute_span_f1(pred_spans, gold_spans):
    """
    Overlap 匹配计算 Span F1
    pred_spans: [(article_id, technique, start, end), ...]
    gold_spans: [(article_id, technique, start, end), ...]
    """
    pool = list(gold_spans)
    tp = fp = 0
    for art_id, tech, ps, pe in pred_spans:
        hit = False
        for i, (g_art, g_tech, gs, ge) in enumerate(pool):
            if g_art == art_id and g_tech == tech and ps < ge and pe > gs:
                tp += 1; pool.pop(i); hit = True; break
        if not hit:
            fp += 1
    fn = len(pool)
    P = tp / (tp + fp) if tp + fp else 0.0
    R = tp / (tp + fn) if tp + fn else 0.0
    F = 2*P*R / (P+R) if P+R else 0.0
    return {'f1': F, 'precision': P, 'recall': R, 'tp': tp, 'fp': fp, 'fn': fn}


print('✅ Helper functions defined')

In [ ]:
# ============================================================
# Cell 5：Sanity Check — 检查 Stage 1 加载是否正确
# ============================================================
# 以 Sup-FT EN 为例，打印前5篇文章的技术集合

path, has_prefix = STAGE1_FILES['Sup-FT']['EN']
stage1_test = load_stage1(path, has_prefix)

print(f'Sup-FT EN：共 {len(stage1_test)} 篇文章有预测')
for art_id, techs in list(stage1_test.items())[:5]:
    print(f'  {art_id}: {techs}')

print(f'\nS2-C EN 过滤前：{len(s2c_preds["EN"])} spans')
filtered_test = filter_s2c_by_stage1(s2c_preds['EN'], stage1_test)
print(f'S2-C EN 过滤后：{len(filtered_test)} spans')

In [ ]:
# ============================================================
# Cell 6：主循环 — 三种 Stage 1 × 三语言
# ============================================================
results = {}

for s1_name, lang_files in STAGE1_FILES.items():
    results[s1_name] = {}
    print(f'\n--- {s1_name} ---')
    for lang, (path, has_prefix) in lang_files.items():
        # 1. 加载 Stage 1 技术集合
        stage1_techs = load_stage1(path, has_prefix)

        # 2. 过滤 S2-C
        filtered = filter_s2c_by_stage1(s2c_preds[lang], stage1_techs)

        # 3. 转换为 tuple 列表
        pred_spans = list(zip(
            filtered['article_id'].astype(str),
            filtered['technique'],
            filtered['start'].astype(int),
            filtered['end'].astype(int),
        ))

        # 4. 加载金标准
        gold_spans = load_gold(GOLD_FILES[lang])

        # 5. 计算 F1
        m = compute_span_f1(pred_spans, gold_spans)
        results[s1_name][lang] = m

        print(f'  {lang}: F1={m["f1"]*100:.2f}%  '
              f'P={m["precision"]*100:.2f}%  '
              f'R={m["recall"]*100:.2f}%  '
              f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}  '
              f'(过滤前 {len(s2c_preds[lang])} → 过滤后 {len(filtered)} spans)')

print('\n✅ Main loop complete')

In [ ]:
# ============================================================
# Cell 7：汇总表格
# ============================================================

# Baselines from Stage 1 × S2-A evaluation（Combo-0：Stage 1 + S2-A LLM，来自 各个方法checkthat2024的结果）
BASELINES = {
    'Sup-FT':   {'EN': (11.82, 7.99,  22.65), 'PL': (8.51, 6.22,  13.50), 'RU': (11.17, 7.45,  22.33)},
    'Prompt-A': {'EN': (7.81,  7.59,  8.05),  'PL': (5.98, 4.75,  8.07),  'RU': (6.13,  4.01,  12.99)},
    'Iter-Ens': {'EN': (10.85, 8.15,  16.21), 'PL': (7.54, 4.76,  18.12), 'RU': (6.16,  3.54,  23.41)},
}

print('=' * 90)
print('实验三汇总：Stage 1 × Stage 2 交叉结果')
print('=' * 90)
print(f'{"Stage 1":<12} {"Stage 2":<14} '
      f'{"EN F1":>8} {"EN P":>7} {"EN R":>7}  '
      f'{"PL F1":>8} {"PL P":>7} {"PL R":>7}  '
      f'{"RU F1":>8} {"RU P":>7} {"RU R":>7}')
print('-' * 90)

for s1_name in ['Sup-FT', 'Prompt-A', 'Iter-Ens']:
    # 基线行
    bl = BASELINES[s1_name]
    print(f'{s1_name:<12} {"S2-A LLM":<14} '
          f'{bl["EN"][0]:>7.2f}% {bl["EN"][1]:>6.2f}% {bl["EN"][2]:>6.2f}%  '
          f'{bl["PL"][0]:>7.2f}% {bl["PL"][1]:>6.2f}% {bl["PL"][2]:>6.2f}%  '
          f'{bl["RU"][0]:>7.2f}% {bl["RU"][1]:>6.2f}% {bl["RU"][2]:>6.2f}%')

    # S2-C 行
    r = results[s1_name]
    en, pl, ru = r['EN'], r['PL'], r['RU']
    print(f'{s1_name:<12} {"S2-C mBERT":<14} '
          f'{en["f1"]*100:>7.2f}% {en["precision"]*100:>6.2f}% {en["recall"]*100:>6.2f}%  '
          f'{pl["f1"]*100:>7.2f}% {pl["precision"]*100:>6.2f}% {pl["recall"]*100:>6.2f}%  '
          f'{ru["f1"]*100:>7.2f}% {ru["precision"]*100:>6.2f}% {ru["recall"]*100:>6.2f}%')
    print()

In [ ]:
# Evaluate S2-C without any Stage 1 filter
import io, boto3, pandas as pd
from botocore.client import Config
from collections import defaultdict

# Reuse helper functions from this notebook（load_gold, compute_span_f1）

s3 = boto3.client(
    's3', endpoint_url="https://your-s3-endpoint"  # Set to your S3-compatible endpoint,
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    config=Config(signature_version='s3v4'),
)

S2C_KEYS = {
    'EN': 'multi_label_models/exp1_s2c_mbert_l10_multilingual/predictions_en.tsv',
    # 'PL': 'multi_label_models/exp1_s2c_mbert_l10_multilingual/predictions_pl.tsv',
    'PL': 'multi_label_models/exp1_s2c_mbert_l10_pl_fixed/predictions_pl.tsv',
    'RU': 'multi_label_models/exp1_s2c_mbert_l10_multilingual/predictions_ru.tsv',
}

GOLD_BASE = "your/path/here"
GOLD_FILES = {
    'EN': GOLD_BASE + 'en_overlap_articles/labels-subtask-3-spans.txt',
    'PL': GOLD_BASE + 'po_overlap_articles/labels-subtask-3-spans.txt',
    'RU': GOLD_BASE + 'ru_overlap_articles/labels-subtask-3-spans.txt',
}

print('=== S2-C 无 Stage 1 过滤（直接全文预测）===')
for lang, key in S2C_KEYS.items():
    buf = io.BytesIO()
    s3.download_fileobj("your-s3-bucket", key, buf)
    buf.seek(0)
    df = pd.read_csv(buf, sep='\t')
    df['article_id'] = df['article_id'].astype(str).str.replace('article', '', regex=False)

    pred_spans = list(zip(df['article_id'], df['technique'],
                          df['start'].astype(int), df['end'].astype(int)))
    gold_spans = load_gold(GOLD_FILES[lang])
    m = compute_span_f1(pred_spans, gold_spans)
    print(f'{lang}: F1={m["f1"]*100:.2f}%  P={m["precision"]*100:.2f}%  R={m["recall"]*100:.2f}%  '
          f'TP={m["tp"]} FP={m["fp"]} FN={m["fn"]}')